In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd

from bs4 import BeautifulSoup

from time import sleep

from datetime import datetime

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

import datetime

import os

from selenium.webdriver.chrome.service import Service as ChromeService



# %%

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'NZ RBNZ'

print(f"Running{regulatorName} Web Scraping Tool v.1.2")


now=datetime.datetime.now()

filename= 'NZ RBNZ Data {}.xlsx'.format(str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') 



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


RunningNZ RBNZ Web Scraping Tool v.1.2


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

In [4]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# Define a function to scroll to the bottom of the page

def scroll_to_bottom(driver):

    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")



    while True:

        # Scroll down to the bottom

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        # Wait to load the page

        sleep(3)
        # Calculate new scroll height and compare with last scroll height

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break

        last_height = new_height

def click_element_by_xpath(driver, xpath):

    # Find the element and click

    element = driver.find_element(By.XPATH, xpath)

    element.click()

def scrollinAndClick(xpath,key_press=False):
    if len(xpath) != 0 :
        for times in range(60):
            try:
                driver.find_element(By.XPATH, xpath).click()
                sleep(1)
                break
            except:
                print(f"[ERROR] : trying {times+1}/10 to key press 'DOWN' (scrolling)")
                sleep(1)
                if key_press:                    
                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)
        else:   
            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')


In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------



regdict={

        'NZ RBNZ 3': 'https://www.rbnz.govt.nz/regulation-and-supervision/banks/register',
        'NZ RBNZ 4': 'https://www.rbnz.govt.nz/regulation-and-supervision/insurers/licensing/register',
        'NZ RBNZ 5': 'https://www.rbnz.govt.nz/regulation-and-supervision/oversight-of-insurers/resources-for-insurers/cancelled-insurance-licences'
        
        }


sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')




In [6]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    
    ## This project need to be verified robot program, solution is start different browers
    try:
        click_element_by_xpath(driver, '/html/body/div[1]/div/main/div[3]/div/div/div[2]/nav/div/div[2]/section/div/a')
    except:
        driver.quit()
        driver = webdriver.Chrome(options=chromeOptions)
        driver.maximize_window()
        driver.get(regdict[reg])
        
    soup=BeautifulSoup(driver.page_source,"html.parser")
    div=soup.find("table",{"class":"table table--text"})
    sleep(3)
    tbody=div.find("tbody")
    trs=tbody.find_all("tr")
    for tr in trs:
        
        tds=tr.find_all("td")
        print(tds[0])
 
        sqldict["ListProcessDate"].append(processdate)
        sqldict["RegCtry"].append("NZ")
        sqldict["RegCode"].append("RBNZ")
        sqldict["ListCode"].append(reg.split(" ")[-1])
        sqldict["Name"].append(tds[0].text.strip())
        if "Cancellation" in div.text:
            sqldict["CancellationDate"].append(tds[1].text.strip())
            sqldict["RegulationType"].append('Cancelled')
        else:
            sqldict["RegulationDate"].append(tds[1].text.strip())
            sqldict["RegulationType"].append('Supervised')
        sqldict["Cntry"].append("NZ")
        if tds[0].find("a") is not None:             
            sqldict["Website"].append(tds[0].find("a",href=True)["href"])
        for key in sqldict.keys():
                if len(sqldict[key])<len(sqldict["Name"]):
                   sqldict[key].append("")

sqldict = bourange_same_length_array(sqldict)


Working with NZ RBNZ 3
<td> </td>
<td style="width: 40%;">
            ANZ Banking Group (New Zealand) Limited<br/>
            ANZ National Bank Limited<br/>
            ANZ Bank New Zealand Ltd<br/>
<a href="https://www.anz.co.nz/about-us/media-centre/investor-information/" rel="noopener noreferrer" target="_blank">Read their latest disclosure statement</a>
</td>
<td>
            ASB Bank Limited<br/>
<a href="https://www.asb.co.nz/legal/disclosure-statements.html" rel="noopener noreferrer" target="_blank">Read their latest disclosure statement</a>
</td>
<td>
            Australia and New Zealand Banking Group Limited (B)<br/>
<a href="https://www.anz.co.nz/about-us/media-centre/investor-information/nz-branch-limited/ " rel="noopener noreferrer" target="_blank">Read their latest disclosure statement</a>
</td>
<td>
            Baroda (New Zealand) Limited<br/>
            Bank of Baroda (New Zealand) Limited<br/>
<a href="https://www.barodanzltd.co.nz/general-disclosure-statement" rel

In [7]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)


df=pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    

C:\Users\wuj1\AppData\Local\Temp\22\ipykernel_33292\2425337739.py:12: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [8]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
1,,,,,,ANZ Banking Group (New Zealand) Limited\n ...,,,,,...,,,,,,,,,,
2,,,,,,ASB Bank Limited\nRead their latest disclosure...,,,,,...,,,,,,,,,,
3,,,,,,Australia and New Zealand Banking Group Limite...,,,,,...,,,,,,,,,,
4,,,,,,Baroda (New Zealand) Limited\n Bank...,,,,,...,,,,,,,,,,
5,,,,,,Bank of China Limited (B)\nRead their latest d...,,,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,,,,,,Tatua Insurance Limited,,,,,...,,,,,,,,,,
157,,,,,,The Hibernian Catholic Benefit Society,,,,,...,,,,,,,,,,
158,,,,,,The National Mutual Life Association of Austra...,,,,,...,,,,,,,,,,
159,,,,,,TOWER Health & Life Limited,,,,,...,,,,,,,,,,
